# **Analisis Reliabilitas LLM-as-a-Judge pada Evaluasi Bahasa Indonesia**

_An Experimental Study of Prompt Sensitivity and Positional Bias in Indonesian Language Evaluation_

**Author:** Rozhak  
**Last Updated:** 14 November 2026  
**Dataset:** IndoPref  
**Evaluator:** `Qwen3-32B (Non-Thinking Mode)`

**Description**

Pipeline evaluasi *end-to-end* untuk mengukur reliabilitas `Qwen3-32B` (`16-bit BF16`, *non-thinking mode*) sebagai *LLM-as-a-Judge* pada evaluasi *pairwise* berbahasa Indonesia menggunakan dataset IndoPref (519 sampel unik). Pengujian mencakup 3 variasi prompt (`P1`, `P2`, `P3`) dan 2 urutan posisi respons (`AB`, `BA`) dengan total 3.114 inferensi.

**Purpose**

Mengukur kecocokan keputusan model dengan preferensi manusia (*Accuracy*, *Cohen's Kappa*), konsistensi terhadap pertukaran urutan respons (*Position Consistency*, *Preference Flip Rate*, uji *McNemar*), serta stabilitas antar variasi prompt (*Prompt Consistency*, *Prompt Flip Rate*).

**Category:** Natural Language Processing / LLM Evaluation  
**Tags:** `llm`, `evaluation`, `reliability`, `indonesian-nlp`


## **1. Introduction**

### **1.1 Research Background**

Penggunaan *LLM-as-a-Judge* sering terkendala masalah konsistensi, terutama akibat bias urutan respons (*position bias*) dan sensitivitas terhadap variasi prompt (*prompt sensitivity*).

Studi sistematis terkait bias ini umumnya berfokus pada bahasa Inggris, sementara pengujian pada bahasa Indonesia masih terbatas. Eksperimen ini mengevaluasi reliabilitas `Qwen3-32B` (*non-thinking mode*, `16-bit BF16`) pada tugas evaluasi *pairwise* berbahasa Indonesia menggunakan dataset IndoPref dengan membandingkan keputusan model terhadap label preferensi manusia.


### **1.2 Experimental Design**

Eksperimen menguji 519 sampel unik pada **6 kombinasi kondisi** (`3` prompt $\times$ `2` posisi), menghasilkan total **3.114 inferensi**:

| Faktor Uji | Konfigurasi | Tujuan Pengujian |
|---|---|---|
| **Urutan Posisi** | `AB` (asli), `BA` (terbalik) | Mengukur konsistensi model terhadap pertukaran posisi (*position bias*). |
| **Variasi Prompt** | `P1` (Baseline), `P2` (Multi-criteria), `P3` (RCAF Constraint) | Mengukur sensitivitas keputusan terhadap perbedaan instruksi (*prompt sensitivity*). |


### **1.3 Evaluation Metrics**

- **Human Alignment:**
  - `Accuracy`: Persentase kecocokan keputusan model dengan label preferensi manusia.
  - `Cohen's Kappa`: Tingkat kesepakatan setelah memperhitungkan faktor kebetulan (*chance agreement*).
- **Positional Reliability:**
  - `Position Consistency` & `Preference Flip Rate`: Persentase keputusan yang tetap sama vs berubah saat urutan `AB` ditukar ke `BA`.
  - `McNemar Test`: Uji signifikansi statistik ($\alpha = 0.05$) terhadap *position bias*.
- **Prompt Stability:**
  - `Prompt Consistency` & `Prompt Flip Rate`: Stabilitas keputusan antar variasi prompt secara berpasangan (*pairwise*) dan sekaligus (*3-way*).


### **1.4 Pipeline Workflow**

1. **Environment Setup:** Instalasi dependensi, mengecek GPU, *mount* Google Drive, dan login Hugging Face.
2. **Dataset Preparation:** Memuat IndoPref, validasi skema, dan deduplikasi menjadi `519` sampel unik (`indopref_eval_stratified.parquet`).
3. **Model Loading:** Memuat *tokenizer* dan bobot `Qwen3-32B` dalam presisi `16-bit BF16` ke GPU.
4. **Prompt Engineering:** Terapkan *template* `P1`–`P3` ke *chat template* Qwen3 (*non-thinking mode*).
5. **Model Inference:** Eksekusi `3.114` inferensi *pairwise* dengan *checkpointing* ke `main_outputs_final.jsonl`.
6. **Metric Evaluation:** *Parsing* output (`A`/`B`), hitung metrik reliabilitas & uji McNemar, lalu ekspor tabel `CSV`/`Parquet`.
7. **Result Visualization:** Ekspor 5 figur evaluasi ke format `PNG` dan `SVG` (`300 DPI`).


## **2. Environment Setup**

Instalasi dependensi, verifikasi GPU, konfigurasi direktori kerja, serta autentikasi Google Drive dan Hugging Face.


### **2.1 Dependency Installation**

Menginstal dan mengimpor dependensi utama untuk pemrosesan data (`datasets`, `pandas`, `pyarrow`), inferensi model (`torch`, `transformers`, `accelerate`), uji statistik (`scikit-learn`, `statsmodels`), serta visualisasi (`matplotlib`, `seaborn`).


In [ ]:
!pip install -q datasets pyarrow pandas matplotlib seaborn transformers accelerate scikit-learn statsmodels

In [ ]:
import datetime
import json
import os
import gc
import re
import time
import shutil
from typing import Tuple, List, Dict, Any, Optional

import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch

from datasets import load_dataset
from sklearn.model_selection import train_test_split

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.contingency_tables import mcnemar

print(
    f"[ENVIRONMENT] status=READY | "
    f"dependencies=torch,transformers,datasets,accelerate,scikit-learn | "
    f"libraries=IMPORTED"
)

### **2.2 Hardware and Runtime Verification**

Mengonfigurasi direktori kerja, `RANDOM_SEED`, `MODEL_ID`, serta memeriksa ketersediaan GPU dan kapasitas VRAM. Direktori dataset dan hasil disimpan di Google Drive (`PROJECT_ROOT`), sedangkan `HF_HOME` diarahkan ke `/content/models` pada penyimpanan lokal untuk menghindari *I/O bottleneck* saat memuat bobot model.


In [ ]:
PROJECT_ROOT: str = '/content/drive/MyDrive/LLMReliability'
DATA_RAW_DIR: str = os.path.join(PROJECT_ROOT, 'data', 'raw')
DATA_PROCESSED_DIR: str = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR: str = '/content/models'
RESULTS_DIR: str = os.path.join(PROJECT_ROOT, 'results')
MODEL_ID: str = 'Qwen/Qwen3-32B'

RANDOM_SEED: int = 42

os.environ['HF_HOME'] = MODELS_DIR
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'

try:
    from transformers.utils import logging as hf_logging
    hf_logging.disable_progress_bar()
except Exception:
    pass

if torch.cuda.is_available():
    device_name: str = torch.cuda.get_device_name(0)
    vram_gb: float = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    device_info: str = f"CUDA | gpu='{device_name}' | vram={vram_gb:.2f} GB"
else:
    device_info: str = "CPU | warning=GPU_NOT_DETECTED"

print(
    f"[ENVIRONMENT] device={device_info} | "
    f"seed={RANDOM_SEED} | "
    f"model_id='{MODEL_ID}' | "
    f"status=CONFIGURED"
)

### **2.3 Storage and Authentication**

Melakukan *mount* Google Drive (dengan *fallback* otomatis untuk *local runtime*), autentikasi Hugging Face via `HF_TOKEN` di `secrets.json`, dan inisialisasi direktori kerja (`DATA_RAW_DIR`, `DATA_PROCESSED_DIR`, `MODELS_DIR`, `RESULTS_DIR`).


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')

    secrets_path: str = '/content/drive/MyDrive/secrets.json'
    if os.path.exists(secrets_path):
        with open(secrets_path, 'r', encoding='utf-8') as f:
            secrets = json.load(f)
        hf_token: str = secrets.get('HF_TOKEN', '')
        if hf_token:
            from huggingface_hub import login
            login(token=hf_token)
            print("[ENVIRONMENT] status=AUTHENTICATED | component='HuggingFace'")
        else:
            print("[ENVIRONMENT] status=WARNING | component='HF_TOKEN not found in secrets.json'")
    else:
        print(f"[ENVIRONMENT] status=WARNING | component='secrets.json not found at {secrets_path}'")
except ImportError:
    print("[ENVIRONMENT] status=LOCAL | component='Google Drive'")

for d in [DATA_RAW_DIR, DATA_PROCESSED_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("[ENVIRONMENT] status=READY | directories='all_initialized'")

## **3. Dataset Preparation**

Memuat dataset IndoPref, validasi skema, deduplikasi sampel, dan menyimpan dataset evaluasi ke format Parquet.


### **3.1 Dataset Ingestion**

Memuat dataset `davidanugraha/IndoPref` dengan *caching* bertingkat: memeriksa *Silver cache* (`indopref_eval_stratified.parquet`) terlebih dahulu, lalu *Bronze cache* (`indopref_raw.parquet`, validasi `> 519` baris), dan mengunduh *split* `test` dari Hugging Face Hub hanya jika kedua *cache* tidak tersedia atau tidak valid.


In [ ]:
eval_path: str = os.path.join(DATA_PROCESSED_DIR, 'indopref_eval_stratified.parquet')
raw_path: str = os.path.join(DATA_RAW_DIR, 'indopref_raw.parquet')

EVAL_SAMPLE_SIZE: Optional[int] = None
FORCE_REGENERATE_DATA: bool = False

df_raw: Optional[pd.DataFrame] = None
df_clean: Optional[pd.DataFrame] = None
df_eval: Optional[pd.DataFrame] = None

use_existing_silver: bool = False
if os.path.exists(eval_path) and not FORCE_REGENERATE_DATA:
    cached_silver = pd.read_parquet(eval_path)
    size_matched = (
        (EVAL_SAMPLE_SIZE is None and len(cached_silver) >= 500) or
        (EVAL_SAMPLE_SIZE is not None and len(cached_silver) == EVAL_SAMPLE_SIZE)
    )
    if size_matched:
        df_eval = cached_silver
        use_existing_silver = True
        print(f"[DATASET] status=CACHE_HIT | tier='processed_silver' | samples={len(df_eval)} | path='{eval_path}'")
    else:
        target_str = 'FULL (>=500)' if EVAL_SAMPLE_SIZE is None else str(EVAL_SAMPLE_SIZE)
        print(f"[DATASET] status=CACHE_INVALIDATED | reason='sample_size_changed' | cached={len(cached_silver)} | target={target_str}")

if not use_existing_silver:
    raw_cache_valid: bool = False
    if os.path.exists(raw_path):
        cached_raw = pd.read_parquet(raw_path)
        if len(cached_raw) > 519:
            df_raw = cached_raw
            raw_cache_valid = True
            print(f"[DATASET] status=CACHE_HIT | tier='raw_bronze' | records={len(df_raw)} | path='{raw_path}'")
        else:
            print(f"[DATASET] status=CACHE_INVALIDATED | tier='raw_bronze' | reason='contaminated_clean_rows' | cached={len(cached_raw)}")

    if not raw_cache_valid:
        print("[DATASET] status=CACHE_MISS | tier='remote_fetch' | id='davidanugraha/IndoPref'")
        dataset = load_dataset('davidanugraha/IndoPref', split='test')
        df_raw = dataset.to_pandas()
        df_raw.to_parquet(raw_path, index=False)
        print(f"[DATASET] status=PERSISTED_BRONZE | raw_records={len(df_raw)} | path='{raw_path}'")

### **3.2 Schema Validator Definition**

Mendefinisikan `validate_dataset_schema` untuk memverifikasi keberadaan enam kolom wajib (`original_id`, `domain`, `input`, `response_1`, `response_2`, `actual_score`) serta memastikan tidak ada nilai *null* maupun *string* kosong pada kolom teks.


In [ ]:
def validate_dataset_schema(df: pd.DataFrame) -> bool:
    required_cols: List[str] = ['original_id', 'domain', 'input', 'response_1', 'response_2', 'actual_score']
    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        print(f"[VALIDATION] status=FAILED | error='missing_columns' | columns={missing}")
        return False
    if df[required_cols].isnull().sum().sum() > 0:
        print("[VALIDATION] status=FAILED | error='null_values'")
        return False
    for col in ['input', 'response_1', 'response_2']:
        if (df[col].astype(str).str.strip() == '').any():
            print(f"[VALIDATION] status=FAILED | error='empty_string' | column='{col}'")
            return False

    print("[VALIDATION] status=PASSED | schema='valid'")
    return True

### **3.3 Schema Integrity Verification**

Menjalankan `validate_dataset_schema` pada dataset aktif (`df_eval` atau `df_raw`) dengan mekanisme *fail-fast* (`ValueError`) jika validasi gagal.


In [ ]:
target_df_to_check = df_eval if df_eval is not None else df_raw
if not validate_dataset_schema(target_df_to_check):
    raise ValueError("[VALIDATION] error='schema_verification_failed'")

print(
    f"[VALIDATION] dataset_shape={target_df_to_check.shape} | "
    f"required_columns={len(['original_id', 'domain', 'input', 'response_1', 'response_2', 'actual_score'])} | "
    f"null_counts=0 | "
    f"empty_strings=0 | "
    f"status=PASSED"
)

### **3.4 Sample Deduplication**

Menghapus baris duplikat pada `df_raw` berdasarkan `original_id` agar setiap instruksi unik hanya dievaluasi satu kali (dilewati jika data dimuat dari *Silver cache*).


In [ ]:
if df_eval is None and df_raw is not None:
    duplicates: int = int(df_raw.duplicated(subset=['original_id']).sum())
    if duplicates > 0:
        df_clean = df_raw.drop_duplicates(subset=['original_id']).copy()
        print(f"[VALIDATION] status=DEDUPLICATED | raw_records={len(df_raw)} | removed={duplicates} | clean_records={len(df_clean)}")
    else:
        df_clean = df_raw.copy()
        print(f"[VALIDATION] status=CLEAN | records={len(df_clean)}")
else:
    print("[VALIDATION] status=SKIPPED_DEDUP | reason='Loaded from pre-validated silver cache'")

### **3.5 Evaluation Cohort Sampling**

Menyiapkan dataset evaluasi final (`df_eval`) dan menyimpannya ke *Silver cache* (`indopref_eval_stratified.parquet`). Secara *default* seluruh sampel unik digunakan (`EVAL_SAMPLE_SIZE=None`); jika dibatasi, *sampling* dilakukan secara *stratified* berdasarkan `domain`.


In [ ]:
if df_eval is None and df_clean is not None:
    if EVAL_SAMPLE_SIZE is not None and len(df_clean) > EVAL_SAMPLE_SIZE:
        df_eval, _ = train_test_split(
            df_clean,
            train_size=EVAL_SAMPLE_SIZE,
            stratify=df_clean['domain'],
            random_state=RANDOM_SEED
        )
        print(f"[DATASET] status=PERSISTED_SILVER | cohort_mode=subsample | samples={len(df_eval)} | stratify=domain | path='{eval_path}'")
    else:
        df_eval = df_clean.copy()
        print(f"[DATASET] status=PERSISTED_SILVER | cohort_mode=full_census | samples={len(df_eval)} | domain_balanced=True | path='{eval_path}'")
    df_eval.to_parquet(eval_path, index=False)
else:
    print(f"[DATASET] status=REUSED_SILVER | cohort_mode=full_census | samples={len(df_eval)} | path='{eval_path}'")

### **3.6 Dataset Inspection and Sample Profiling**

Menampilkan ringkasan `df_eval` (jumlah sampel, penggunaan memori, distribusi `domain`) serta satu contoh baris lengkap untuk memverifikasi format `input`, `response_1`, `response_2`, dan `actual_score`.


In [ ]:
display_cols: List[str] = ['original_id', 'domain', 'input', 'response_1', 'response_2', 'actual_score']
print(f"[DATASET_INSPECTION] total_samples={len(df_eval)} | memory_usage={df_eval.memory_usage().sum() / 1024:.1f} KB")
display(df_eval[display_cols].head(3))

print("[DATASET_DISTRIBUTION] Komposisi Sampel per Domain:")
domain_dist = df_eval['domain'].value_counts().to_frame(name='Jumlah Sampel')
domain_dist['Persentase (%)'] = (domain_dist['Jumlah Sampel'] / len(df_eval) * 100).round(2)
display(domain_dist)

sample_row = df_eval.iloc[0]
raw_actual = str(sample_row.get('actual_score', 'N/A'))
print(f"[SAMPLE_METADATA] id={sample_row['original_id']} | domain='{sample_row['domain']}'")
print(f"[SAMPLE_INPUT] {sample_row['input']}")
print(f"[SAMPLE_RESPONSE_1] length={len(sample_row['response_1'])} chars | text:\n{sample_row['response_1']}")
print(f"[SAMPLE_RESPONSE_2] length={len(sample_row['response_2'])} chars | text:\n{sample_row['response_2']}")
print(f"[SAMPLE_GROUND_TRUTH] actual_score='{raw_actual}'")

## **4. Model Loading and Precision Setup**

Memuat *tokenizer* dan bobot `Qwen3-32B` ke GPU dalam presisi `bfloat16` (`16-bit`).


### **4.1 Tokenizer Initialization**

Memuat *tokenizer* `Qwen3-32B` menggunakan direktori *cache* `MODELS_DIR` dan memverifikasi ukuran *vocabulary*.


In [ ]:
print(f"[MODEL] id='{MODEL_ID}' | status=LOADING_TOKENIZER")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=MODELS_DIR)
print(f"[MODEL] status=TOKENIZER_READY | vocab_size={tokenizer.vocab_size}")

### **4.2 16-Bit BF16 Precision Configuration**

Menentukan `MODEL_DTYPE` (`torch.bfloat16` jika didukung GPU, atau *fallback* ke `torch.float16`) dan mengaktifkan `TF32` untuk mempercepat operasi matriks. Konfigurasi ini mempertahankan presisi penuh `16-bit` tanpa kuantisasi.


In [ ]:
bf16_supported: bool = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
MODEL_DTYPE: torch.dtype = torch.bfloat16 if bf16_supported else torch.float16

if bf16_supported:
    torch.backends.cuda.matmul.allow_tf32 = True

print(
    f"[PRECISION] dtype={MODEL_DTYPE} | "
    f"bf16_supported={bf16_supported} | "
    f"quantization='none_native_16bit' | "
    f"status=CONFIGURED"
)

### **4.3 Weight Loading and Memory Profiling**

Memuat bobot `Qwen3-32B` menggunakan `MODEL_DTYPE` dan `device_map='auto'`, lalu mencatat penggunaan VRAM GPU (`memory_allocated` dan `memory_reserved`).


In [ ]:
print(f"[MODEL] id='{MODEL_ID}' | status=LOADING_MODEL | dtype={MODEL_DTYPE} | cache_dir='{MODELS_DIR}'")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=MODEL_DTYPE,
    device_map='auto',
    cache_dir=MODELS_DIR
)

allocated_vram: float = torch.cuda.memory_allocated() / (1024**3)
reserved_vram: float = torch.cuda.memory_reserved() / (1024**3)

print(f"[MODEL] status=LOADED | device={model.device} | dtype={model.dtype} | allocated_vram={allocated_vram:.2f} GB | reserved_vram={reserved_vram:.2f} GB")

## **5. Prompt Engineering**

Perancangan variasi *prompt template*, penerapan *chat template* (*non-thinking mode*), dan inspeksi hasil tokenisasi.


### **5.1 Prompt Strategy Definition**

Mendefinisikan `format_prompt` dengan tiga skema prompt yang sama-sama mewajibkan output satu huruf (`A` atau `B`) tanpa penjelasan:

| ID | Skema | Fokus Instruksi |
|---|---|---|
| `P1` | Baseline | Penilaian *pairwise* secara menyeluruh untuk memilih respons terbaik. |
| `P2` | Multi-criteria | Penilaian terstruktur berdasarkan Akurasi, Relevansi, dan Kejelasan. |
| `P3` | RCAF Constraint | Penilaian dengan larangan eksplisit terhadap bias posisi dan bias panjang respons (*verbosity*). |


In [ ]:
def format_prompt(input_text: str, resp_A: str, resp_B: str, p_id: str) -> str:
    if p_id == 'P1':
        return (
            f"### TUGAS\nBandingkan dua respons AI di bawah ini untuk instruksi yang sama. Pilih mana yang secara keseluruhan lebih baik.\n\n"
            f"### INSTRUKSI\n{input_text}\n\n"
            f"### RESPONS A\n{resp_A}\n\n"
            f"### RESPONS B\n{resp_B}\n\n"
            f"### FORMAT JAWABAN\nKeluarkan HANYA satu huruf: A atau B. Dilarang menulis alasan atau teks lain."
        )
    elif p_id == 'P2':
        return (
            f"### TUGAS\nNilai dua respons AI berikut berdasarkan: Akurasi (kebenaran), Relevansi (menjawab instruksi), dan Kejelasan (tidak bertele-tele).\n\n"
            f"### INSTRUKSI\n{input_text}\n\n"
            f"### RESPONS A\n{resp_A}\n\n"
            f"### RESPONS B\n{resp_B}\n\n"
            f"### FORMAT JAWABAN\nBerdasarkan kriteria di atas, keluarkan HANYA satu huruf: A atau B. Dilarang memberikan penjelasan."
        )
    elif p_id == 'P3':
        return (
            f"### TUGAS\nTentukan respons terbaik untuk instruksi di bawah ini.\n\n"
            f"### INSTRUKSI\n{input_text}\n\n"
            f"### RESPONS A\n{resp_A}\n\n"
            f"### RESPONS B\n{resp_B}\n\n"
            f"### ATURAN KETAT\n- Abaikan urutan kemunculan respons (jangan bias posisi).\n- Abaikan panjang respons (jangan bias verbosity).\n- FORMAT JAWABAN WAJIB: Keluarkan hanya karakter A atau B."
        )
    return ""

print("[PROMPT_ENGINEERING] strategies=['P1_Baseline', 'P2_MultiCriteria', 'P3_RCAF'] | status=FUNCTIONS_DEFINED")

### **5.2 Chat Template Formatting**

Mendefinisikan `apply_chat_template` untuk memformat prompt ke struktur *system/user message* Qwen3 dengan `enable_thinking=False` (*non-thinking mode*), sehingga model langsung menghasilkan keputusan akhir tanpa proses *thinking* internal.


In [ ]:
def apply_chat_template(tokenizer, text: str, enable_thinking: bool = False) -> str:
    messages = [
        {'role': 'system', 'content': 'Anda adalah evaluator AI yang objektif. Anda tidak memiliki bias posisi. Jawab HANYA dengan pilihan kandidat (A atau B).'},
        {'role': 'user', 'content': text}
    ]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

print("[CHAT_TEMPLATE] model='Qwen3-32B' | mode='non_thinking' | add_generation_prompt=True | status=CONFIGURED")

### **5.3 Concrete Prompt Inspection**

Menguji `format_prompt` dan `apply_chat_template` pada satu sampel (`P1` urutan `AB`/`BA` dan `P3` urutan `AB`) untuk memverifikasi struktur prompt final serta jumlah token input sebelum inferensi massal.


In [ ]:
demo_sample = df_eval.iloc[0]
demo_input: str = demo_sample['input']
resp_1: str = demo_sample['response_1']
resp_2: str = demo_sample['response_2']

prompt_p1_ab: str = format_prompt(demo_input, resp_1, resp_2, p_id='P1')
chat_p1_ab: str = apply_chat_template(tokenizer, prompt_p1_ab)
tokens_ab: int = len(tokenizer.encode(chat_p1_ab))

prompt_p1_ba: str = format_prompt(demo_input, resp_2, resp_1, p_id='P1')
chat_p1_ba: str = apply_chat_template(tokenizer, prompt_p1_ba)
tokens_ba: int = len(tokenizer.encode(chat_p1_ba))

print(f"[PROMPT_INSPECTION] condition='AB' | strategy='P1' | tokens={tokens_ab}")
print(chat_p1_ab)
print(f"[PROMPT_INSPECTION] condition='BA' | strategy='P1' | tokens={tokens_ba}")
print(chat_p1_ba)

prompt_p3_ab: str = format_prompt(demo_input, resp_1, resp_2, p_id='P3')
chat_p3_ab: str = apply_chat_template(tokenizer, prompt_p3_ab)
tokens_p3: int = len(tokenizer.encode(chat_p3_ab))
print(f"[PROMPT_INSPECTION] condition='AB' | strategy='P3_RCAF' | tokens={tokens_p3} | status=VERIFIED")

## **6. Model Inference**

Eksekusi 3.114 inferensi *pairwise* (`519` sampel $\times$ `3` prompt $\times$ `2` posisi) dengan penyimpanan bertahap (*checkpointing*) ke Google Drive.


### **6.1 Inference Progress Tracking**

Mendefinisikan `log_progress` untuk mencatat progres eksekusi (persentase selesai, waktu berjalan, dan *throughput* `tokens/sec`) setiap kelipatan 15 inferensi (`log_interval=15`).


In [ ]:
def log_progress(current: int, total: int, session_start: float, log_interval: int = 15, **kwargs):
    if (current % log_interval == 0) or (current >= total):
        elapsed: float = time.time() - session_start
        pct: float = (current / total) * 100.0 if total > 0 else 0.0
        metrics_str: str = ' | '.join([f"{k}={v}" for k, v in kwargs.items()])
        print(f"[EVALUATION] {pct:.1f}% ({current}/{total}) | elapsed={elapsed:.1f}s | {metrics_str}")

print("[EVALUATION] status=LOGGER_READY | log_interval=15")

### **6.2 Symmetric Paired Evaluation**

Menjalankan *inference loop* pada seluruh kombinasi `(sample_id, prompt_id, position)` yang belum tercatat di `main_outputs_final.jsonl` sehingga mendukung *resume* otomatis jika sesi terputus. Setiap tugas dieksekusi dengan *greedy decoding* (`do_sample=False`, `max_new_tokens=10`), langsung ditulis per baris (`flush()`), diikuti pembersihan *cache* VRAM (`torch.cuda.empty_cache()`).


In [ ]:
out_file: str = os.path.join(RESULTS_DIR, 'main_outputs_final.jsonl')
prompts: List[str] = ['P1', 'P2', 'P3']
positions: List[str] = ['AB', 'BA']

def determine_human_winner(row_data: Any) -> str:
    if 'actual_score' in row_data:
        val: str = str(row_data['actual_score']).strip().upper()
        if 'ASSISTANT A' in val or val in ['A', '1', '1.0']:
            return 'A'
        if 'ASSISTANT B' in val or val in ['B', '2', '2.0']:
            return 'B'
        try:
            parsed_val = json.loads(str(row_data['actual_score']).replace("'", '"'))
            if isinstance(parsed_val, (list, tuple)) and len(parsed_val) >= 2:
                return 'A' if float(parsed_val[0]) >= float(parsed_val[1]) else 'B'
        except Exception:
            pass
    if 'model_1' in row_data and 'chosen_model' in row_data:
        if str(row_data['chosen_model']).strip() == str(row_data['model_1']).strip():
            return 'A'
        elif 'rejected_model' in row_data and str(row_data['rejected_model']).strip() == str(row_data['model_1']).strip():
            return 'B'
    if 'label' in row_data:
        l_val: str = str(row_data['label']).strip().upper()
        if 'ASSISTANT A' in l_val or l_val in ['1', '0', 'A']:
            return 'A'
        if 'ASSISTANT B' in l_val or l_val in ['2', 'B']:
            return 'B'
    return 'A'

In [ ]:
processed_combinations: set = set()
if os.path.exists(out_file):
    with open(out_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                record_item = json.loads(line)
                processed_combinations.add((record_item['sample_id'], record_item['prompt_id'], record_item['position']))

target_tasks: List[Dict[str, Any]] = []
for _, row in df_eval.iterrows():
    s_id: int = int(row['original_id'])
    for p_id in prompts:
        for pos in positions:
            target_tasks.append({
                'row': row,
                'sample_id': s_id,
                'prompt_id': p_id,
                'position': pos,
                'key': (s_id, p_id, pos)
            })

total_cohort_tasks: int = len(target_tasks)
pending_tasks: List[Dict[str, Any]] = [t for t in target_tasks if t['key'] not in processed_combinations]
already_completed: int = total_cohort_tasks - len(pending_tasks)

print(f"[RESUME] cohort_total={total_cohort_tasks} | already_completed={already_completed} | pending={len(pending_tasks)}")
print(f"[INFERENCE_CONFIG] sampling=False | max_new_tokens=10 | temperature=0.0 | format='JSONL' | sink='{out_file}'")

In [ ]:
if not pending_tasks:
    print("[INFERENCE] status=ALREADY_COMPLETED | All cohort tasks exist in results.")
else:
    print(f"[INFERENCE] status=STARTED | remaining_evaluations={len(pending_tasks)}")
    current_step: int = already_completed
    inference_start_t: float = time.time()
    with torch.no_grad():
        for task in pending_tasks:
            current_step += 1
            row = task['row']
            sample_id = task['sample_id']
            p_id = task['prompt_id']
            pos = task['position']

            human_winner: str = determine_human_winner(row)
            if pos == 'AB':
                resp_A, resp_B = row['response_1'], row['response_2']
                current_human = human_winner
            else:
                resp_A, resp_B = row['response_2'], row['response_1']
                current_human = 'B' if human_winner == 'A' else 'A'

            start_t: float = time.time()
            raw_prompt: str = format_prompt(row['input'], resp_A, resp_B, p_id)
            formatted_prompt: str = apply_chat_template(tokenizer, raw_prompt)
            inputs = tokenizer(formatted_prompt, return_tensors='pt').to(model.device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

            gen_tokens = outputs[0][inputs['input_ids'].shape[1]:]
            raw_out: str = tokenizer.decode(gen_tokens, skip_special_tokens=True)
            runtime_sec: float = round(time.time() - start_t, 2)
            tps: float = round(len(gen_tokens) / runtime_sec, 2) if runtime_sec > 0 else 0.0

            record: Dict[str, Any] = {
                'sample_id': sample_id,
                'domain': row['domain'],
                'prompt_id': p_id,
                'position': pos,
                'human_preference': current_human,
                'raw_output': raw_out,
                'runtime_sec': runtime_sec,
                'output_tokens': len(gen_tokens),
                'tps': tps
            }

            with open(out_file, 'a', encoding='utf-8') as f:
                f.write(json.dumps(record) + '\n')
                f.flush()

            processed_combinations.add((sample_id, p_id, pos))
            log_progress(current_step, total_cohort_tasks, inference_start_t, log_interval=15, ID=sample_id, Prompt=p_id, Pos=pos, TPS=tps)

            del inputs, outputs
            torch.cuda.empty_cache()
            gc.collect()

print(f"[INFERENCE] status=COMPLETED | sink='{out_file}'")

## **7. Reliability Metric Evaluation**

Ekstraksi keputusan model, pemetaan ulang posisi, perhitungan metrik reliabilitas, uji signifikansi McNemar, dan analisis stabilitas prompt.


### **7.1 Decision Extraction and Mapping**

Memuat hasil inferensi dari `main_outputs_final.jsonl`, menyinkronkan ulang label `human_preference` terhadap `df_eval`, serta mengekstrak pilihan `A`/`B` menggunakan *regex* (`parse_decision`). Pilihan huruf dipetakan ke indeks respons asli (`1` atau `2`) pada `logical_decision`, lalu data dideduplikasi dan difilter ke `df_valid` dengan mengecualikan output `INVALID`.


In [ ]:
def parse_decision(raw_output: str) -> str:
    clean = str(raw_output).strip().upper()
    match = re.search(r'\b([AB])\b', clean)
    return match.group(1) if match else 'INVALID'

In [ ]:
records: List[Dict[str, Any]] = []
with open(out_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

if 'df_eval' in globals() and df_eval is not None:
    eval_gt_map = {int(r['original_id']): determine_human_winner(r) for _, r in df_eval.iterrows()}
    records_updated: bool = False
    for r in records:
        s_id = int(r['sample_id'])
        if s_id in eval_gt_map:
            true_winner = eval_gt_map[s_id]
            expected_human = true_winner if r['position'] == 'AB' else ('B' if true_winner == 'A' else 'A')
            if r.get('human_preference') != expected_human:
                r['human_preference'] = expected_human
                records_updated = True
    if records_updated:
        with open(out_file, 'w', encoding='utf-8') as f:
            for r in records:
                f.write(json.dumps(r) + '\n')
        print(f"[RECONCILIATION] status=SYNCHRONIZED | sink='{out_file}' | ground_truth_aligned=True")

for r in records:
    parsed: str = parse_decision(r['raw_output'])
    r['parsed_decision'] = parsed

    if parsed == 'INVALID':
        r['logical_decision'] = 'INVALID'
    elif r['position'] == 'AB':
        r['logical_decision'] = 1 if parsed == 'A' else 2
    else:
        r['logical_decision'] = 2 if parsed == 'A' else 1

    if r['position'] == 'AB':
        r['logical_human'] = 1 if r['human_preference'] == 'A' else 2
    else:
        r['logical_human'] = 2 if r['human_preference'] == 'A' else 1

df_all = pd.DataFrame(records).drop_duplicates(subset=['sample_id', 'prompt_id', 'position'], keep='last')
invalid_count: int = int((df_all['parsed_decision'] == 'INVALID').sum())
failure_rate: float = (invalid_count / len(df_all)) * 100

print(f"[PARSER] total_records={len(df_all)} | invalid={invalid_count} | failure_rate={failure_rate:.2f}%")

df_valid = df_all[df_all['parsed_decision'] != 'INVALID'].copy()

### **7.2 Positional Consistency Evaluation**

Memasangkan `logical_decision` antara urutan `AB` dan `BA` per `(sample_id, prompt_id)` untuk menghitung *Position Consistency*, *Flip Rate*, serta proporsi pemilihan posisi `A` (*first-position selection rate*) sebagai indikator awal *primacy bias*.


In [ ]:
df_pos = df_valid.pivot_table(
    index=['sample_id', 'prompt_id'],
    columns='position',
    values='logical_decision',
    aggfunc='first'
).dropna()

position_consistency: float = float((df_pos['AB'] == df_pos['BA']).mean() * 100)
position_flip: float = 100.0 - position_consistency
first_pos_rate: float = float((df_valid['parsed_decision'] == 'A').mean() * 100)

print(f"[METRICS] overall_consistency={position_consistency:.2f}%")
print(f"[METRICS] overall_flip_rate={position_flip:.2f}%")
print(f"[METRICS] first_position_selection_rate={first_pos_rate:.2f}%")

### **7.3 McNemar Statistical Significance Test**

Menjalankan uji McNemar dengan koreksi kontinuitas ($\alpha = 0.05$) pada kasus inkonsisten antara urutan `AB` dan `BA`: membandingkan frekuensi model yang selalu memilih posisi pertama (`A` di kedua urutan) terhadap posisi kedua (`B` di kedua urutan) untuk menentukan signifikansi statistik dari *position bias*.


In [ ]:
b: int = int(sum((df_pos['AB'] == 1) & (df_pos['BA'] == 2)))
c: int = int(sum((df_pos['AB'] == 2) & (df_pos['BA'] == 1)))

table = [[0, b], [c, 0]]
mcnemar_res = mcnemar(table, exact=False, correction=True)

is_significant: bool = bool(mcnemar_res.pvalue < 0.05)
conclusion_str: str = 'SIGNIFICANT_POSITION_BIAS_DETECTED' if is_significant else 'NO_SIGNIFICANT_BIAS'

print(
    f"[STATISTICS] test='McNemar' | "
    f"statistic={mcnemar_res.statistic:.4f} | "
    f"pvalue={mcnemar_res.pvalue:.5e} | "
    f"alpha=0.05 | "
    f"conclusion='{conclusion_str}'"
)

### **7.4 Human Preference Alignment**

Menghitung *Accuracy* dan *Cohen's Kappa* terhadap preferensi manusia untuk tiap prompt (`P1`, `P2`, `P3`) beserta *95% bootstrap confidence interval* (500 iterasi), mencatat performa *baseline* (`P1_AB`), serta mengekspor tabel ringkasan dan detail evaluasi ke format CSV dan Parquet.


In [ ]:
def bootstrap_kappa(y_true: list, y_pred: list, n_iterations: int = 500, alpha: float = 0.05, seed: int = RANDOM_SEED) -> Tuple[float, float]:
    rng = np.random.default_rng(seed)
    stats = []
    yt, yp = np.array(y_true), np.array(y_pred)
    n = len(yt)

    for _ in range(n_iterations):
        idx = rng.integers(0, n, n)
        stats.append(cohen_kappa_score(yt[idx], yp[idx]))

    return float(np.percentile(stats, (alpha/2)*100)), float(np.percentile(stats, (1-alpha/2)*100))

In [ ]:
metrics_summary = []
for p_id in prompts:
    sub = df_valid[df_valid['prompt_id'] == p_id]
    acc = (sub['logical_decision'] == sub['logical_human']).mean() * 100
    sub_pos = df_pos.xs(p_id, level='prompt_id') if p_id in df_pos.index.levels[1] else pd.DataFrame()
    consistency = (sub_pos['AB'] == sub_pos['BA']).mean() * 100 if not sub_pos.empty else 0.0

    kappa = cohen_kappa_score(list(sub['logical_decision']), list(sub['logical_human']))
    ci_low, ci_high = bootstrap_kappa(list(sub['logical_human']), list(sub['logical_decision']), n_iterations=500)

    metrics_summary.append({
        'Prompt': p_id,
        'Accuracy (%)': round(acc, 2),
        'Consistency (%)': round(consistency, 2),
        'Flip Rate (%)': round(100 - consistency, 2),
        'Cohen Kappa': round(kappa, 3),
        'Kappa 95% CI': f"[{ci_low:.3f}, {ci_high:.3f}]"
    })

df_metrics_table = pd.DataFrame(metrics_summary)
display(df_metrics_table)

sub_baseline = df_valid[(df_valid['prompt_id'] == 'P1') & (df_valid['position'] == 'AB')]
if not sub_baseline.empty:
    base_acc = (sub_baseline['logical_decision'] == sub_baseline['logical_human']).mean() * 100
    base_kappa = cohen_kappa_score(list(sub_baseline['logical_decision']), list(sub_baseline['logical_human']))

    print(f"[BASELINE] condition='P1_AB' | accuracy={base_acc:.2f}% | cohen_kappa={base_kappa:.3f}")

In [ ]:
metrics_csv: str = os.path.join(RESULTS_DIR, 'evaluation_metrics_summary.csv')
metrics_parquet: str = os.path.join(RESULTS_DIR, 'evaluation_metrics_summary.parquet')
df_metrics_table.to_csv(metrics_csv, index=False)
df_metrics_table.to_parquet(metrics_parquet, index=False)

print(f"[ARTIFACT] table='Evaluation Metrics Summary' | formats=['CSV', 'Parquet'] | dir='{RESULTS_DIR}' | status=SAVED")

parsed_results_path: str = os.path.join(RESULTS_DIR, 'evaluation_results_detailed.parquet')
df_valid.to_parquet(parsed_results_path, index=False)

print(f"[ARTIFACT] table='Evaluation Results Detailed' | format='Parquet' | records={len(df_valid)} | path='{parsed_results_path}' | status=SAVED")

### **7.5 Prompt Stability Analysis**

Mengukur stabilitas keputusan model antar variasi prompt pada urutan `AB` untuk kesepakatan *3-way* (`P1 = P2 = P3`) dan perbandingan *pairwise* (`P1 vs P2`, `P1 vs P3`, `P2 vs P3`), lalu mengekspor `prompt_stability_summary.csv` dan `executive_summary_kpi.csv`.


In [ ]:
prompt_consistency_3way: float = 0.0
prompt_flip_3way: float = 0.0

df_prompt_stab = df_valid[df_valid['position'] == 'AB'].pivot_table(
    index='sample_id',
    columns='prompt_id',
    values='logical_decision'
).dropna()

if not df_prompt_stab.empty:
    prompt_consistency_3way = float(((df_prompt_stab['P1'] == df_prompt_stab['P2']) & (df_prompt_stab['P2'] == df_prompt_stab['P3'])).mean() * 100)
    prompt_flip_3way = 100.0 - prompt_consistency_3way

    p1_p2_flip: float = float((df_prompt_stab['P1'] != df_prompt_stab['P2']).mean() * 100)
    p1_p3_flip: float = float((df_prompt_stab['P1'] != df_prompt_stab['P3']).mean() * 100)
    p2_p3_flip: float = float((df_prompt_stab['P2'] != df_prompt_stab['P3']).mean() * 100)

    prompt_stability_records = [
        {'Comparison': 'P1=P2=P3 (Simultaneous 3-Way)', 'Consistency (%)': round(prompt_consistency_3way, 2), 'Flip Rate (%)': round(prompt_flip_3way, 2)},
        {'Comparison': 'P1 vs P2 (Pairwise)', 'Consistency (%)': round(100.0 - p1_p2_flip, 2), 'Flip Rate (%)': round(p1_p2_flip, 2)},
        {'Comparison': 'P1 vs P3 (Pairwise)', 'Consistency (%)': round(100.0 - p1_p3_flip, 2), 'Flip Rate (%)': round(p1_p3_flip, 2)},
        {'Comparison': 'P2 vs P3 (Pairwise)', 'Consistency (%)': round(100.0 - p2_p3_flip, 2), 'Flip Rate (%)': round(p2_p3_flip, 2)}
    ]
    df_prompt_stability_summary = pd.DataFrame(prompt_stability_records)
    display(df_prompt_stability_summary)

    prompt_stab_csv: str = os.path.join(RESULTS_DIR, 'prompt_stability_summary.csv')
    df_prompt_stability_summary.to_csv(prompt_stab_csv, index=False)
    print(f"[ARTIFACT] table='Prompt Stability Summary' | format='CSV' | path='{prompt_stab_csv}' | status=SAVED")

exec_kpi_records = [
    {'Metric Category': 'Cohort Population', 'Specification': 'IndoPref Sensus Prompt Unik', 'Value': f"{len(df_eval)} samples ({len(df_valid):,} evaluations)"},
    {'Metric Category': 'Positional Reliability', 'Specification': 'Consistency (AB == BA)', 'Value': f"{position_consistency:.2f}%"},
    {'Metric Category': 'Positional Reliability', 'Specification': 'Preference Flip Rate', 'Value': f"{position_flip:.2f}%"},
    {'Metric Category': 'Positional Bias Test', 'Specification': 'McNemar Test (alpha=0.05)', 'Value': f"p = {mcnemar_res.pvalue:.2e} (Significant)" if is_significant else f"p = {mcnemar_res.pvalue:.2e} (Not Significant)"},
    {'Metric Category': 'Prompt Stability', 'Specification': 'Simultaneous 3-Way Consistency', 'Value': f"{prompt_consistency_3way:.2f}%"},
    {'Metric Category': 'Prompt Stability', 'Specification': 'Simultaneous 3-Way Flip Rate', 'Value': f"{prompt_flip_3way:.2f}%"}
]
if 'base_acc' in locals() or 'base_acc' in globals():
    exec_kpi_records.append({
        'Metric Category': 'Baseline Performance',
        'Specification': 'P1_AB Condition',
        'Value': f"Accuracy = {base_acc:.2f}% | Kappa = {base_kappa:.3f}"
    })

df_executive_summary = pd.DataFrame(exec_kpi_records)
display(df_executive_summary)

exec_summary_csv: str = os.path.join(RESULTS_DIR, 'executive_summary_kpi.csv')
df_executive_summary.to_csv(exec_summary_csv, index=False)
print(f"[ARTIFACT] table='Executive Summary KPI' | format='CSV' | path='{exec_summary_csv}' | status=SAVED")
print(f"[PIPELINE] status=COMPLETED | cohort={len(df_eval)} | instances={len(df_valid)} | sink='{RESULTS_DIR}'")

## **8. Result Visualization**

Pembuatan dan ekspor lima figur evaluasi (`300 DPI`, format PNG dan SVG) yang merangkum akurasi, *position bias*, tingkat keberhasilan *parsing*, dan stabilitas prompt.


### **8.1 Visualization Setup**

Konfigurasi tema global Matplotlib/Seaborn (`whitegrid`, `300 DPI`) serta fungsi *helper* untuk anotasi nilai (`annotate_bars`), *bar chart* (`plot_bars`), dan ekspor per panel ke PNG dan SVG (`export_panels`).


In [ ]:
sns.set_theme(style='whitegrid', font_scale=1.15)
plt.rcParams.update({
    'font.sans-serif': 'DejaVu Sans', 'font.family': 'sans-serif',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 10.5, 'ytick.labelsize': 10.5,
    'legend.fontsize': 9.5, 'figure.dpi': 300, 'savefig.dpi': 300
})

def annotate_bars(ax: plt.Axes, bars=None, yerr=None, fmt: str = "{:.2f}%", horizontal: bool = False) -> None:
    for i, b in enumerate(bars if bars is not None else ax.patches):
        val = b.get_width() if horizontal else b.get_height()
        span = b.get_height() if horizontal else b.get_width()
        if not np.isnan(val) and val > 0 and span > 0:
            err = yerr[i] if yerr is not None and i < len(yerr) else 0.0
            xy = (val + err, b.get_y() + span / 2) if horizontal else (b.get_x() + span / 2, val + err)
            ax.annotate(fmt.format(val), xy, xytext=(4, 0) if horizontal else (0, 4),
                        textcoords='offset points', ha='left' if horizontal else 'center',
                        va='center' if horizontal else 'bottom', fontsize=9.5 if horizontal else 10,
                        fontweight='bold', zorder=5,
                        bbox=dict(boxstyle='square,pad=0.15', fc='white', ec='none', alpha=0.9))

def plot_bars(ax: plt.Axes, x, y, palette: str, title: str, xlabel: str, ylabel: str,
              ylim: Tuple[float, float] = (0, 100), yerr=None, fmt: str = "{:.2f}%") -> None:
    bars = ax.bar(x, y, width=0.55, yerr=yerr, capsize=5 if yerr is not None else 0,
                  color=sns.color_palette(palette, len(x)), edgecolor='black', alpha=0.85)
    ax.set(xlabel=xlabel, ylabel=ylabel, ylim=ylim)
    ax.set_title(title, fontweight='bold', pad=10)
    err_top = yerr[1] if isinstance(yerr, list) and len(yerr) == 2 and isinstance(yerr[1], list) else yerr
    annotate_bars(ax, bars=bars, yerr=err_top, fmt=fmt)

def export_panels(fig: plt.Figure, axes, names: List[str], results_dir: str = RESULTS_DIR, dpi: int = 300) -> None:
    fig.tight_layout()
    ax_list = list(np.atleast_1d(axes))
    for target_ax, base_name in zip(ax_list, names):
        orig_title = target_ax.get_title()
        target_ax.set_title(re.sub(r'^[A-Z]\.\s+', '', orig_title), fontweight='bold', pad=10)
        for ax in ax_list:
            ax.set_visible(ax is target_ax)
        full_base = os.path.join(results_dir, base_name)
        fig.savefig(f"{full_base}.png", dpi=dpi, bbox_inches='tight')
        fig.savefig(f"{full_base}.svg", format='svg', bbox_inches='tight')
        target_ax.set_title(orig_title, fontweight='bold', pad=10)
        print(f"[VISUALIZATION] figure='{base_name}' | formats=['PNG', 'SVG'] | dir='{results_dir}' | status=SAVED")
    for ax in ax_list:
        ax.set_visible(True)
    plt.show()

print("[ENVIRONMENT] visualization_framework='Matplotlib/Seaborn' | theme='whitegrid' | base_dpi=300 | status=CONFIGURED")

### **8.2 Figure 1: Accuracy and Position Distribution**

Visualisasi dua panel yang membandingkan akurasi model tiap prompt terhadap *baseline* acak 50% beserta *95% confidence interval* (Panel A), serta distribusi proporsi pilihan `A` dan `B` pada urutan `AB` dan `BA` untuk mendeteksi *position bias* (Panel B).


In [ ]:
fig1, axes1 = plt.subplots(1, 2, figsize=(14.5, 4.6))

acc_s = (df_valid.assign(ok=(df_valid['logical_decision'] == df_valid['logical_human']) * 100)
         .groupby('prompt_id')['ok'].agg(['mean', 'count']).reindex(prompts).fillna(0))
acc_err = (1.96 * np.sqrt(acc_s['mean'] * (100 - acc_s['mean']) / acc_s['count'].replace(0, np.nan)).fillna(0)).tolist()

plot_bars(axes1[0], prompts, acc_s['mean'].tolist(), 'Blues_d',
          'A. Model Accuracy against Human Preference', 'Prompt Strategy', 'Accuracy (%)', yerr=acc_err)
axes1[0].axhline(50.0, color='gray', ls='--', lw=1.5, zorder=1, label='Chance Baseline (50%)')
axes1[0].legend(loc='upper right')

df_pos_dist = (df_valid.groupby(['position', 'parsed_decision']).size().unstack(fill_value=0)
               .reindex(columns=['A', 'B'], fill_value=0))
df_pos_dist = (df_pos_dist.div(df_pos_dist.sum(axis=1), axis=0) * 100).reset_index().melt(id_vars='position', var_name='Slot', value_name='Pct')
df_pos_dist['Slot'] = df_pos_dist['Slot'].map({'A': 'Position A (1st)', 'B': 'Position B (2nd)'})

sns.barplot(data=df_pos_dist, x='Slot', y='Pct', hue='position', ax=axes1[1], palette='Set2', edgecolor='black', alpha=0.85)
axes1[1].axhline(50.0, color='gray', ls='--', lw=1.5, zorder=1, label='Ideal Unbiased Split (50%)')
axes1[1].set(xlabel='Displayed Position Slot', ylabel='Selection Proportion (%)', ylim=(0, 100))
axes1[1].set_title('B. Positional Slot Selection Distribution', fontweight='bold', pad=10)
axes1[1].legend(title='Presentation Order', loc='upper right')
annotate_bars(axes1[1], fmt="{:.1f}%")

export_panels(fig1, axes1, ['plot_model_accuracy', 'plot_slot_selection_distribution'])

### **8.3 Figure 2: Consistency and Cohen Kappa Profile**

Perbandingan metrik reliabilitas antar prompt dalam tiga panel: *Position Consistency* (Panel A), *Flip Rate* (Panel B), dan *Cohen's Kappa* beserta ambang batas *Moderate* (`0.40`) dan *Substantial* (`0.60`) serta *95% CI* (Panel C).


In [ ]:
fig2, axes2 = plt.subplots(1, 3, figsize=(18.0, 4.6))

ci = df_metrics_table['Kappa 95% CI'].str.strip('[]').str.split(',', expand=True).astype(float)
k_vals = df_metrics_table['Cohen Kappa'].astype(float)
k_err = [(k_vals - ci[0]).clip(lower=0).tolist(), (ci[1] - k_vals).clip(lower=0).tolist()]

plot_bars(axes2[0], df_metrics_table['Prompt'], df_metrics_table['Consistency (%)'], 'Blues_d',
          'A. Positional Consistency (Higher is Better)', 'Prompt Strategy', 'Consistency Rate (%)')
plot_bars(axes2[1], df_metrics_table['Prompt'], df_metrics_table['Flip Rate (%)'], 'Reds_d',
          'B. Decision Flip Rate (Lower is Better)', 'Prompt Strategy', 'Flip Rate (%)')
plot_bars(axes2[2], df_metrics_table['Prompt'], k_vals.tolist(), 'Greens_d',
          "C. Cohen's Kappa Agreement with Human Preference", 'Prompt Strategy', "Cohen's Kappa Coefficient",
          ylim=(0.0, 1.15), yerr=k_err, fmt="{:.3f}")
axes2[2].axhline(0.40, color='darkgreen', ls=':', lw=1.5, zorder=1, label='Moderate Agreement (0.40)')
axes2[2].axhline(0.60, color='navy', ls='--', lw=1.2, zorder=1, label='Substantial Agreement (0.60)')
axes2[2].legend(loc='upper right', fontsize=9)

export_panels(fig2, axes2, ['plot_positional_consistency', 'plot_positional_flip_rate', 'plot_cohen_kappa'])

### **8.4 Figure 3: Semantic Parsing and Domain Accuracy**

Visualisasi rasio validitas format output (`A`/`B` vs `INVALID`) dalam bentuk *donut chart* (Panel A) dan akurasi model per `domain` secara terurut beserta rata-rata keseluruhan, sekaligus mengekspor `domain_accuracy_summary.csv` (Panel B).


In [ ]:
fig3, axes3 = plt.subplots(1, 2, figsize=(15.0, 4.8), gridspec_kw={'width_ratios': [1.0, 1.35]})

total_eval = len(df_valid) + invalid_count
valid_pct = (len(df_valid) / total_eval) * 100 if total_eval > 0 else 0.0
invalid_pct = (invalid_count / total_eval) * 100 if total_eval > 0 else 0.0

axes3[0].pie([len(df_valid), invalid_count] if invalid_count > 0 else [len(df_valid)],
             colors=['#27ae60', '#e74c3c'] if invalid_count > 0 else ['#27ae60'],
             startangle=90, wedgeprops=dict(width=0.4, edgecolor='white', lw=2))
axes3[0].text(0, 0, f"{valid_pct:.1f}%\nValid\n(N={total_eval:,})", ha='center', va='center', fontweight='bold', fontsize=11.5)
axes3[0].set_title('A. Decision Format Parsing Success Ratio', fontweight='bold', pad=10)
axes3[0].legend(handles=[
    Patch(facecolor='#27ae60', label=f"Valid (A/B): {len(df_valid):,} ({valid_pct:.1f}%)"),
    Patch(facecolor='#e74c3c', label=f"Invalid: {invalid_count:,} ({invalid_pct:.1f}%)")
], loc='lower center', bbox_to_anchor=(0.5, -0.1), ncol=2, fontsize=9)

df_valid['is_correct'] = (df_valid['logical_decision'] == df_valid['logical_human']).astype(int) * 100
domain_acc = df_valid.groupby('domain')['is_correct'].mean().reset_index().sort_values('is_correct')
mean_acc = float(df_valid['is_correct'].mean())
domain_csv: str = os.path.join(RESULTS_DIR, 'domain_accuracy_summary.csv')
domain_acc.sort_values('is_correct', ascending=False).to_csv(domain_csv, index=False)
print(f"[DATASET] table='Domain Accuracy Summary' | path='{domain_csv}' | status=EXPORTED")

bars_dom = axes3[1].barh(domain_acc['domain'], domain_acc['is_correct'],
                         color=sns.color_palette('viridis', len(domain_acc)), edgecolor='black', alpha=0.85)
axes3[1].axvline(mean_acc, color='red', ls='--', lw=1.5, zorder=1, label=f'Overall Mean ({mean_acc:.1f}%)')
axes3[1].set(xlabel='Accuracy Rate (%)', ylabel='Topical Domain', xlim=(0, 105))
axes3[1].set_title('B. Model Evaluator Accuracy across Topical Domains', fontweight='bold', pad=10)
axes3[1].legend(loc='lower right', fontsize=9)
annotate_bars(axes3[1], bars=bars_dom, fmt="{:.1f}%", horizontal=True)

export_panels(fig3, axes3, ['plot_parsing_success_ratio', 'plot_domain_accuracy'])

### **8.5 Figure 4: Positional Decision Heatmap**

*Decision matrix heatmap* $2 \times 2$ untuk tiap prompt (`P1`, `P2`, `P3`) yang memetakan keputusan model pada urutan `AB` terhadap `BA` berdasarkan identitas respons asli (`Resp. 1`, `Resp. 2`) guna melihat konsistensi pada diagonal utama.


In [ ]:
fig4, axes4 = plt.subplots(1, len(prompts), figsize=(16.5, 4.6))

for ax, p_id in zip(axes4, prompts):
    sub_p = df_valid[df_valid['prompt_id'] == p_id]
    paired = (sub_p[sub_p['position'] == 'AB'].set_index('sample_id')
              .join(sub_p[sub_p['position'] == 'BA'].set_index('sample_id'), lsuffix='_ab', rsuffix='_ba', how='inner'))
    ct = pd.crosstab(paired['logical_decision_ab'], paired['logical_decision_ba']).reindex(index=[1, 2], columns=[1, 2], fill_value=0)
    ct.index = ct.columns = ['Resp. 1', 'Resp. 2']
    n_pair = max(len(paired), 1)
    annot = [[f"{v}\n({v / n_pair * 100:.1f}%)" for v in row] for row in ct.to_numpy()]
    cons = (np.diag(ct).sum() / len(paired) * 100) if len(paired) > 0 else 0.0

    sns.heatmap(ct, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=False, square=True,
                annot_kws={'fontsize': 11, 'fontweight': 'bold'}, linewidths=1, linecolor='gray')
    ax.set(xlabel='Choice in BA Order (Reversed)', ylabel='Choice in AB Order (Standard)')
    ax.set_title(f"{p_id}: Positional Matrix (Consistent: {cons:.1f}%)", fontweight='bold', pad=10)
    ax.tick_params(axis='y', rotation=0)

export_panels(fig4, axes4, [f"plot_positional_heatmap_{p.lower()}" for p in prompts])

### **8.6 Figure 5: Cross-Prompt Stability and Sensitivity Profile**

Perbandingan stabilitas keputusan antar pasangan prompt (`P1 vs P2`, `P1 vs P3`, `P2 vs P3`) pada urutan `AB` dalam dua panel: *Prompt Consistency* (Panel A) dan *Prompt Flip Rate* (Panel B).


In [ ]:
if 'df_prompt_stab' not in globals() or df_prompt_stab is None or df_prompt_stab.empty:
    df_prompt_stab = df_valid[df_valid['position'] == 'AB'].pivot_table(
        index='sample_id', columns='prompt_id', values='logical_decision'
    ).dropna()

if not df_prompt_stab.empty:
    pairs = [('P1 vs P2', 'P1', 'P2'), ('P1 vs P3', 'P1', 'P3'), ('P2 vs P3', 'P2', 'P3')]
    labels = [lbl for lbl, _, _ in pairs]
    cons_rates = [float((df_prompt_stab[a] == df_prompt_stab[b]).mean() * 100) for _, a, b in pairs]
    flip_rates = [100.0 - c for c in cons_rates]

    fig5, axes5 = plt.subplots(1, 2, figsize=(14.5, 4.6))
    plot_bars(axes5[0], labels, cons_rates, 'Blues_d', 'A. Cross-Prompt Decision Consistency (Higher is Better)', 'Prompt Pairwise Comparison', 'Consistency Rate (%)')
    plot_bars(axes5[1], labels, flip_rates, 'Reds_d', 'B. Cross-Prompt Decision Flip Rate (Lower is Better)', 'Prompt Pairwise Comparison', 'Flip Rate (%)')
    export_panels(fig5, axes5, ['plot_prompt_consistency', 'plot_prompt_flip_rate'])